In [ ]:
## This script prepares data for the Networks training
## It combines the data from 2 channels of an antenna to make an event which is used as input to CNNs.
## Test Train split is also done in this script

from IPython.display import display, HTML
display(HTML("<style>.container { width:85% !important; }</style>"))

In [1]:
import os

ABS_PATH_HERE = str(os.getcwd())

# 2D Denoiser data creation

In [2]:
import numpy as np

from sklearn.model_selection import train_test_split
import os

# Directory where the data is stored
DataDir = ABS_PATH_HERE + "/data/sample_set/"

# Loading sig+noise data
ns_ch0 = np.load(DataDir + "/ch0_SigPlusNoise.npz")
ns_ch0 = ns_ch0["arr_0"]
ns_ch1 = np.load(DataDir + "/ch1_SigPlusNoise.npz")
ns_ch1 = ns_ch1["arr_0"]

# Loading pure signal data
sig_ch0 = np.load(DataDir + "/ch0_Signals.npz")
sig_ch0 = sig_ch0["arr_0"]
sig_ch1 = np.load(DataDir + "/ch1_Signals.npz")
sig_ch1 = sig_ch1["arr_0"]

print(f"Initial shapes = sig: {sig_ch0.shape}, sig_plus_noise: {ns_ch0.shape}")

# Combine two channels to make an event
sig_only = np.stack([sig_ch0, sig_ch1], axis=2)  
sig_noise = np.stack([ns_ch0, ns_ch1], axis=2)

print(f"Concate shape Sig_noise = {sig_noise.shape} and signals only = {sig_only.shape}")

del ns_ch0, ns_ch1, sig_ch0, sig_ch1

## Split the test and train data (30 : 70)
Traces_train, Traces_test, labels_train, labels_test= train_test_split(sig_noise,
                                                             sig_only,
                                                             random_state=42,
                                                             test_size=0.3)

print("Saving TstTrain data")

Outpath = os.path.join(ABS_PATH_HERE, "data", "TstTrain_Denoiser")

# Create the directory (and parents) only if it doesn't already exist
os.makedirs(Outpath, exist_ok=True)


np.save(Outpath + f"/Noisy_train.npy", Traces_train)
np.save(Outpath + f"/Noisy_test.npy", Traces_test)
np.save(Outpath + f"/Signals_train.npy", labels_train)
np.save(Outpath + f"/Signals_test.npy", labels_test)

del Traces_train, Traces_test, labels_train, labels_test

Initial shapes = sig: (500, 1000), sig_plus_noise: (500, 1000)
Concate shape Sig_noise = (500, 1000, 2) and signals only = (500, 1000, 2)
Saving TstTrain data


# 2D Classifier data creation

In [3]:
## Here we combine the channels data and create labels for the classifier

# Loading data

ns_ch0 = np.load(DataDir + "/ch0_SigPlusNoise.npz")
ns_ch0 = ns_ch0["arr_0"]
ns_ch1 = np.load(DataDir + "/ch1_SigPlusNoise.npz")
ns_ch1 = ns_ch1["arr_0"]

no_ch0 = np.load(DataDir + "/ch0_NoiseOnly.npz")
no_ch0 = no_ch0["arr_0"]
no_ch1 = np.load(DataDir + "/ch1_NoiseOnly.npz")
no_ch1 = no_ch1["arr_0"]

print(f"Initial shapes = sig: {ns_ch0.shape}, noise: {no_ch0.shape}")

noise_only = np.stack([no_ch0, no_ch1], axis=2) ### Combine two channels to make an event 
sig_noise = np.stack([ns_ch0, ns_ch1], axis=2)

print(f"Concate shape Sig_noise = {sig_noise.shape} and noise only = {noise_only.shape}")

del ns_ch0, ns_ch1, no_ch0, no_ch1

# Labels for signal (1) and background (0) traces
# We create one label per event
L1 = np.ones(len(sig_noise))
L2 = np.zeros(len(noise_only))

Traces = np.concatenate((sig_noise, noise_only))

Labels = np.concatenate((L1, L2))

del sig_noise, noise_only, L1, L2

## Splitting the data
Traces_train, Traces_test, Labels_train, Labels_test = train_test_split(Traces, Labels,
                                                    random_state=42,
                                                    test_size=0.3)

print("Saving TstTrain data")

OutpathClassifier = os.path.join(ABS_PATH_HERE, "data", "TstTrain_Classifier")

# Create the directory (and parents) only if it doesn't already exist
os.makedirs(OutpathClassifier, exist_ok=True)

### Saving the data
np.save(OutpathClassifier + f"/Traces_train.npy", Traces_train)
np.save(OutpathClassifier + f"/Traces_test.npy", Traces_test)
np.save(OutpathClassifier + f"/Labels_train.npy", Labels_train)
np.save(OutpathClassifier + f"/Labels_test.npy", Labels_test)


del Traces_train, Traces_test, Labels_train, Labels_test

Initial shapes = sig: (500, 1000), noise: (500, 1000)
Concate shape Sig_noise = (500, 1000, 2) and noise only = (500, 1000, 2)
Saving TstTrain data
